# 面试问题：SGD、Momentum、Adam 与 AdamW 的更新规则和适用场景是什么？

可以直接复述的回答是：SGD 只沿当前梯度下降，更新透明但在病态曲率上容易震荡。Momentum 用速度累计历史方向，能穿过狭长谷底。Adam 同时维护梯度一阶矩与二阶矩，并对早期矩估计做 bias correction，因此不同尺度参数能获得自适应步长。AdamW 把权重衰减从梯度中解耦，避免正则项被二阶矩重新缩放。实践中要在同一数据和训练预算下比较收敛曲线，同时明确 bias、归一化参数通常不衰减。下面不调用 `torch.optim`，逐项写出四种更新并观察真实参数轨迹。

## 真实案例：根据温度与客流预测门店小时销量

八条脱敏门店快照包含温度、小时客流和实际销量。数值结构仿照零售短期预测，但不是线上数据；小样本只用于观察优化器机制，不能外推实际收益。

In [1]:
import warnings  # 导入告警控制以保持教学输出整洁
import torch  # 导入 PyTorch 张量和自动微分能力
warnings.filterwarnings("ignore")  # 隐藏环境相关但不影响实验的告警
torch.set_num_threads(1)  # 固定单线程以减少教学实验抖动
torch.manual_seed(7)  # 固定参数初始化和计算随机性
records = [  # 定义八个带业务语义的门店小时快照
    ("浦东午间", 31.0, 260.0, 221.0),  # 高温高客流门店样本
    ("静安早间", 22.0, 110.0, 111.0),  # 温和低客流门店样本
    ("徐汇晚间", 27.0, 230.0, 199.0),  # 晚高峰门店样本
    ("杨浦午间", 29.0, 180.0, 163.0),  # 中等客流门店样本
    ("虹口早间", 18.0, 80.0, 83.0),  # 低温低客流门店样本
    ("长宁晚间", 25.0, 200.0, 177.0),  # 稳定晚间门店样本
    ("闵行午间", 33.0, 300.0, 251.0),  # 最高客流门店样本
    ("普陀早间", 20.0, 95.0, 98.0),  # 普通早间门店样本
]  # 结束门店快照列表
x_raw = torch.tensor([[row[1], row[2]] for row in records], dtype=torch.float32)  # 提取温度和客流特征
y_raw = torch.tensor([row[3] for row in records], dtype=torch.float32)  # 提取实际小时销量
print("输入预览：场景 | 温度 | 客流 | 实际销量")  # 输出原始业务字段标题
for row in records:  # 逐条展示八个真实语义样本
    print(f"{row[0]:6} | {row[1]:4.1f} | {row[2]:5.1f} | {row[3]:5.1f}")  # 展示可读输入记录
print("张量形状：", tuple(x_raw.shape), tuple(y_raw.shape))  # 展示模型接收的张量形状

输入预览：场景 | 温度 | 客流 | 实际销量
浦东午间   | 31.0 | 260.0 | 221.0
静安早间   | 22.0 | 110.0 | 111.0
徐汇晚间   | 27.0 | 230.0 | 199.0
杨浦午间   | 29.0 | 180.0 | 163.0
虹口早间   | 18.0 |  80.0 |  83.0
长宁晚间   | 25.0 | 200.0 | 177.0
闵行午间   | 33.0 | 300.0 | 251.0
普陀早间   | 20.0 |  95.0 |  98.0
张量形状： (8, 2) (8,)


## Baseline / 基线：未标准化特征直接跑 SGD

客流量约为温度的十倍，统一学习率会先被客流梯度支配。基线在原始尺度上用保守学习率训练 120 步，记录损失和参数。

In [2]:
def mean_squared_error(prediction, target):  # 手写均方误差供所有优化器共享
    return ((prediction - target) ** 2).mean()  # 返回样本平均平方误差
def train_raw_sgd(steps, learning_rate):  # 在未标准化数据上执行最简单 SGD
    theta = torch.zeros(3, requires_grad=True)  # 初始化温度系数、客流系数和偏置
    trace = []  # 保存关键步损失与参数轨迹
    for step in range(1, steps + 1):  # 重复完整批次前向和反向传播
        prediction = x_raw @ theta[:2] + theta[2]  # 使用当前参数预测原始销量
        loss = mean_squared_error(prediction, y_raw)  # 计算原始尺度训练损失
        loss.backward()  # 真实执行自动微分得到三个参数梯度
        with torch.no_grad():  # 关闭更新操作的梯度记录
            theta -= learning_rate * theta.grad  # 按 SGD 规则直接更新参数
        theta.grad.zero_()  # 清空本步梯度防止意外累积
        if step in {1, 2, 10, steps}:  # 只保留最有解释力的训练节点
            trace.append((step, float(loss), theta.detach().clone()))  # 保存损失和参数快照
    return theta.detach(), trace  # 返回基线参数与过程轨迹
baseline_theta, baseline_trace = train_raw_sgd(120, 0.00001)  # 用同一八样本训练未标准化 SGD
baseline_prediction = x_raw @ baseline_theta[:2] + baseline_theta[2]  # 计算基线逐样本预测
baseline_mae = float((baseline_prediction - y_raw).abs().mean())  # 计算基线平均绝对误差
print("step | loss | [温度系数, 客流系数, 偏置]")  # 输出基线轨迹表头
for step, loss, theta in baseline_trace:  # 逐个展示关键训练节点
    print(f"{step:4d} | {loss:10.2f} | {[round(value, 4) for value in theta.tolist()]}")  # 展示尺度失衡下的参数变化
print(f"基线 MAE={baseline_mae:.3f}")  # 输出可与后续优化器比较的指标

step | loss | [温度系数, 客流系数, 偏置]
   1 |   29769.38 | [0.0889, 0.6786, 0.0033]
   2 |    1410.89 | [0.1084, 0.8216, 0.004]
  10 |      89.61 | [0.1216, 0.8588, 0.0048]
 120 |      78.58 | [0.2305, 0.8447, 0.0123]
基线 MAE=7.657


## 核心实现：标准化后手写四种参数更新

下列实现只让两个特征权重参与 AdamW 的 decoupled weight decay，偏置不衰减。输出前 3 步的梯度、动量或矩估计，以及最终损失。

In [3]:
x_mean = x_raw.mean(dim=0)  # 计算两个特征的训练集均值
x_std = x_raw.std(dim=0, unbiased=False)  # 计算两个特征的训练集标准差
y_mean = y_raw.mean()  # 计算目标均值用于输出反标准化
y_std = y_raw.std(unbiased=False)  # 计算目标标准差用于稳定优化
x_train = (x_raw - x_mean) / x_std  # 将温度和客流变换到相近尺度
y_train = (y_raw - y_mean) / y_std  # 将销量目标标准化
def train_optimizer(name, steps=120, learning_rate=0.08, weight_decay=0.02):  # 手写指定优化器的完整训练循环
    theta = torch.zeros(3, requires_grad=True)  # 为公平比较使用完全相同的初始参数
    first_moment = torch.zeros_like(theta)  # 初始化 Momentum 或 Adam 一阶矩
    second_moment = torch.zeros_like(theta)  # 初始化 Adam 二阶矩
    trace = []  # 保存前三步与末步中间量
    for step in range(1, steps + 1):  # 对同一完整批次重复优化
        prediction = x_train @ theta[:2] + theta[2]  # 在标准化空间执行线性模型前向传播
        loss = mean_squared_error(prediction, y_train)  # 计算标准化均方误差
        loss.backward()  # 真实执行 backward 得到当前梯度
        gradient = theta.grad.detach().clone()  # 复制梯度供更新和轨迹展示
        with torch.no_grad():  # 不让优化器状态更新进入计算图
            if name == "SGD":  # 处理纯随机梯度下降分支
                update = gradient  # SGD 直接使用当前梯度
            elif name == "Momentum":  # 处理带动量的 SGD 分支
                first_moment = 0.9 * first_moment + gradient  # 累积指数衰减速度
                update = first_moment  # 用速度而不是瞬时梯度更新
            else:  # 处理 Adam 与 AdamW 的共同矩估计
                first_moment = 0.9 * first_moment + 0.1 * gradient  # 更新梯度一阶指数滑动平均
                second_moment = 0.999 * second_moment + 0.001 * gradient.square()  # 更新梯度平方滑动平均
                corrected_first = first_moment / (1.0 - 0.9 ** step)  # 修正早期一阶矩偏向零的问题
                corrected_second = second_moment / (1.0 - 0.999 ** step)  # 修正早期二阶矩偏向零的问题
                update = corrected_first / (corrected_second.sqrt() + 1e-8)  # 形成逐参数自适应更新量
            if name == "AdamW":  # 单独处理解耦权重衰减
                theta[:2] *= 1.0 - learning_rate * weight_decay  # 只对特征权重做独立乘法衰减
            theta -= learning_rate * update  # 应用当前优化器计算出的参数更新
        theta.grad.zero_()  # 清空梯度以开始下一训练步
        if step in {1, 2, 3, steps}:  # 选择关键节点保留可解释轨迹
            trace.append({"step": step, "loss": float(loss), "gradient": gradient.clone(), "first": first_moment.clone(), "second": second_moment.clone(), "theta": theta.detach().clone()})  # 保存梯度、状态和参数
    prediction_raw = (x_train @ theta.detach()[:2] + theta.detach()[2]) * y_std + y_mean  # 把预测还原为真实销量单位
    mae = float((prediction_raw - y_raw).abs().mean())  # 计算原始业务单位平均误差
    return theta.detach(), prediction_raw, mae, trace  # 返回参数、预测、指标和轨迹
optimizer_results = {}  # 收集四种优化器的公平比较结果
for optimizer_name in ["SGD", "Momentum", "Adam", "AdamW"]:  # 依次运行四种手写更新规则
    optimizer_results[optimizer_name] = train_optimizer(optimizer_name)  # 保存当前优化器完整实验结果
adam_trace = optimizer_results["Adam"][3]  # 选择 Adam 轨迹解释矩估计
print("Adam 前三步：step | loss | grad | m | v")  # 输出 Adam 中间过程表头
for item in adam_trace[:3]:  # 展示 Adam 最早三个更新节点
    print(f"{item['step']} | {item['loss']:.4f} | {[round(v, 4) for v in item['gradient'].tolist()]} | {[round(v, 4) for v in item['first'].tolist()]} | {[round(v, 5) for v in item['second'].tolist()]}")  # 展示梯度和矩状态如何演化
print("优化器 | 最终 MAE | 标准化参数")  # 输出四种优化器结果表头
for optimizer_name, result in optimizer_results.items():  # 逐个读取统一训练预算下的结果
    print(f"{optimizer_name:8} | {result[2]:8.3f} | {[round(v, 3) for v in result[0].tolist()]}")  # 对比收敛质量和参数

Adam 前三步：step | loss | grad | m | v
1 | 1.0000 | [-1.9026, -1.9993, 0.0] | [-0.1903, -0.1999, 0.0] | [0.00362, 0.004, 0.0]
2 | 0.7175 | [-1.591, -1.6877, -0.137] | [-0.3303, -0.3487, -0.0137] | [0.00615, 0.00684, 2e-05]
3 | 0.4772 | [-1.2821, -1.3787, -0.018] | [-0.4255, -0.4517, -0.0141] | [0.00779, 0.00874, 2e-05]
优化器 | 最终 MAE | 标准化参数
SGD      |    2.340 | [0.209, 0.793, -0.0]
Momentum |    1.067 | [0.042, 0.958, -0.0]
Adam     |    1.358 | [0.083, 0.917, 0.0]
AdamW    |    1.466 | [0.104, 0.895, 0.0]


## 逐样本结果

同一输入和同一 MAE 指标下，标准化后的更新明显优于未标准化基线。这里不宣称 AdamW 永远最好；数据规模、批量噪声和学习率都会改变排序。

In [4]:
best_name = min(optimizer_results, key=lambda name: optimizer_results[name][2])  # 按真实销量 MAE 选择本实验最佳方案
best_prediction = optimizer_results[best_name][1]  # 获取最佳方案逐样本预测
print(f"本教学实验最低 MAE 优化器：{best_name}")  # 展示实验内而非泛化性的选择结果
print("场景 | 实际 | 基线预测 | 优化后预测 | 绝对误差")  # 输出逐样本对照表头
for index, row in enumerate(records):  # 遍历八个门店小时样本
    error = abs(float(best_prediction[index] - y_raw[index]))  # 计算当前样本绝对误差
    print(f"{row[0]:6} | {row[3]:6.1f} | {baseline_prediction[index]:10.1f} | {best_prediction[index]:10.1f} | {error:8.2f}")  # 展示同数据基线与修正结果

本教学实验最低 MAE 优化器：Momentum
场景 | 实际 | 基线预测 | 优化后预测 | 绝对误差
浦东午间   |  221.0 |      226.8 |      221.8 |     0.78
静安早间   |  111.0 |       98.0 |      109.3 |     1.69
徐汇晚间   |  199.0 |      200.5 |      198.2 |     0.76
杨浦午间   |  163.0 |      158.7 |      163.1 |     0.13
虹口早间   |   83.0 |       71.7 |       85.8 |     2.77
长宁晚间   |  177.0 |      174.7 |      175.6 |     1.35
闵行午间   |  251.0 |      261.0 |      251.6 |     0.58
普陀早间   |   98.0 |       84.9 |       97.5 |     0.46


## 失败案例与修正：Adam 忘记 bias correction

第一步的一阶矩只有真实梯度的 0.1 倍，二阶矩只有平方梯度的 0.001 倍。若直接相除，更新会被人为放大；除以对应的累计系数后，第一步更新才与梯度尺度一致。

In [5]:
example_gradient = torch.tensor([0.01, 1.0, 10.0])  # 构造跨三个数量级的首步梯度
example_moment = 0.1 * example_gradient  # 计算 Adam 第一步未修正一阶矩
example_square = 0.001 * example_gradient.square()  # 计算 Adam 第一步未修正二阶矩
uncorrected_update = example_moment / (example_square.sqrt() + 1e-8)  # 复现遗漏偏差修正的错误更新
corrected_moment = example_moment / (1.0 - 0.9)  # 对第一步一阶矩执行偏差修正
corrected_square = example_square / (1.0 - 0.999)  # 对第一步二阶矩执行偏差修正
corrected_update = corrected_moment / (corrected_square.sqrt() + 1e-8)  # 计算正确的首步自适应更新
print("梯度：", example_gradient.tolist())  # 展示三个不同尺度的梯度
print("遗漏 bias correction 的更新：", [round(v, 3) for v in uncorrected_update.tolist()])  # 展示约三倍的错误首步
print("加入 bias correction 的更新：", [round(v, 3) for v in corrected_update.tolist()])  # 展示尺度一致的正确首步

梯度： [0.009999999776482582, 1.0, 10.0]
遗漏 bias correction 的更新： [3.162, 3.162, 3.162]
加入 bias correction 的更新： [1.0, 1.0, 1.0]


## 结果解读

原始特征下，客流梯度控制了步长，温度系数和偏置几乎学不动。标准化后四种算法都能在相同预算下降低 MAE；Adam 的 `m` 和 `v` 明确展示了方向平滑与尺度归一化。Momentum 并不“保存梯度列表”，而是保存一个递推速度；AdamW 也不是把 L2 项塞入梯度，而是在更新时独立衰减参数。

## 生产边界

教学实验是八条全批量 CPU 回归，没有 mini-batch 噪声、学习率调度、稀疏梯度或分布式同步。生产训练还要监控梯度范数、参数组、混合精度溢出和验证集指标，并用框架优化器的经过测试实现。优化器优劣必须与模型、数据、batch size 和预算共同报告。

## 最小回归测试

In [6]:
assert len(records) >= 5  # 保证案例包含足够多的真实业务样本
assert all(torch.isfinite(result[0]).all() for result in optimizer_results.values())  # 保证四种手写优化器参数均为有限值
assert min(result[2] for result in optimizer_results.values()) < baseline_mae  # 保证标准化主方案优于同数据原始尺度基线
assert len(adam_trace) == 4 and adam_trace[0]["step"] == 1  # 保证 Adam 中间轨迹覆盖首步和末步
assert torch.all(corrected_update < uncorrected_update)  # 保证偏差修正确实消除首步放大
assert best_prediction.shape == y_raw.shape  # 保证逐样本预测没有静默广播